## Load API Key

In [1]:
import os
from dotenv import load_dotenv

In [3]:
load_dotenv()

True

In [4]:
key = os.getenv("tmdb_API_key") # should be changed to whatever the key is
print("API key loaded:", key is not None) # check key was loaded correctly

API key loaded: True


## Fetch All Movies of Interest

In [7]:
import requests
import json
import time

In [8]:
# use pagination to request movie lists in any endpoint
def fetch_movies(category, extra_params, pages = 5, delay = 0.5, api_key = key, verbose = True):
    
    base_url = "https://api.themoviedb.org/3/discover/movie" # use discover endpoint to get details of movies
    all_movies= [] # initialize list to store movies
    
    print(f"Fetching {category} movies:")


    for page in range(1, pages + 1):
        
        # define parameters for requests.get() function
        params = {
            'api_key': api_key,
            'page': page,
            **extra_params
        }

        try: # get results from page and save results in all_movies
            response = requests.get(base_url, params = params) # get results of page
            response.raise_for_status() # throw status
            data = response.json() # save data as json

            if 'results' in data:
                all_movies.extend(data['results']) # add elements (append) to the end of the movies list
                if verbose == True:
                    print(f" Page {page}: {len(data['results'])} movies") # print progress
            else:
                print(f"No Results on Page {page}")

        except requests.exceptions.RequestException as e: # stop code from running
            print(f"Error on Page {page}: {e}")
            break
        time.sleep(delay) # ensure api limit

    print(f"Success! Total {category}: {len(all_movies)} movies")

    return all_movies

In [11]:
# define ways to search discover endpoint
categories = {

    # include only top rated movies, defined by top 100 movies with at least 1000 votes
    "Top Rated": {
        "extra_params": {
            "sort_by": "vote_average.desc",
            "vote_count.gte": 1000
        }
    },

    # include medium rated movies, defined by movies with an average rating of 4.5-7.5 and at least 100 votes
    "Medium Rated": {
        "extra_params": {
            "sort_by": "vote_average.desc",
            "vote_average.gte": 4.5,
            "vote_average.lte": 7.5,
            "vote_count.gte": 100
        }
    },

    # include bottom rated movies, defined by movies with an average rating of below 4.5 and at least 100 votes
    "Bottom Rated": {
        "extra_params": {
            "sort_by": "vote_average.asc",
            "vote_average.lte": 4.5,
            "vote_count.gte": 100
        }
    },

    # include most popular movies
    "Most Popular Rated": {
        "extra_params": {
            "sort_by": "popularity.desc",
            "vote_count.gte": 100
        }
    },

    # include least popular movies
    "Least Popular Rated": {
        "extra_params": {
            "sort_by": "popularity.asc",
            "vote_count.gte": 100
        }
    }
}

In [13]:
# define function to go through categories and return movie data
def fetch_all_movies(categories, pages_per_category = 5, delay = 0.5):
    full_dataset = [] # initialize dataset to store all movies

    for name, cfg in categories.items():
        # call helper function to grab movies
        movies = fetch_movies(
            category = name,
            extra_params = cfg['extra_params'],
            pages = pages_per_category,
            delay = delay
        )
        
        full_dataset.extend(movies) # add elements (append) to the end of the movies list

    print(f"Total Movies Across All Categories: {len(full_dataset)}")

    return full_dataset

In [15]:
# uncomment when you want to request data
# all_movies = fetch_all_movies(categories, pages_per_category = 10)

Fetching Top Rated movies:
 Page 1: 20 movies
 Page 2: 20 movies
 Page 3: 20 movies
 Page 4: 20 movies
 Page 5: 20 movies
 Page 6: 20 movies
 Page 7: 20 movies
 Page 8: 20 movies
 Page 9: 20 movies
 Page 10: 20 movies
Success! Total Top Rated: 200 movies
Fetching Medium Rated movies:
 Page 1: 20 movies
 Page 2: 20 movies
 Page 3: 20 movies
 Page 4: 20 movies
 Page 5: 20 movies
 Page 6: 20 movies
 Page 7: 20 movies
 Page 8: 20 movies
 Page 9: 20 movies
 Page 10: 20 movies
Success! Total Medium Rated: 200 movies
Fetching Bottom Rated movies:
 Page 1: 20 movies
 Page 2: 20 movies
 Page 3: 20 movies
 Page 4: 20 movies
 Page 5: 20 movies
 Page 6: 20 movies
 Page 7: 20 movies
 Page 8: 20 movies
 Page 9: 20 movies
 Page 10: 20 movies
Success! Total Bottom Rated: 200 movies
Fetching Most Popular Rated movies:
 Page 1: 20 movies
 Page 2: 20 movies
 Page 3: 20 movies
 Page 4: 20 movies
 Page 5: 20 movies
 Page 6: 20 movies
 Page 7: 20 movies
 Page 8: 20 movies
 Page 9: 20 movies
 Page 10: 20 mov

## Add More Data to Each Movie

In [17]:
def fetch_movie_details(movie_id, api_key = key):
    base_url = f"https://api.themoviedb.org/3/movie/{movie_id}"
    params = {'api_key': api_key}

    try:
        response = requests.get(base_url, params = params)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Unable to Fetch Details for Movie ID {movie_id}: {e}")
        return None

In [19]:
def fetch_all_movie_details(movie_list, delay = 0.5, max_movies = None, api_key = key, verbose = True):
    movies_detailed = []
    for i, movie in enumerate(movie_list):
        if max_movies and i >= max_movies:
            break

        movie_id = movie.get('id')
        if movie_id:
            details = fetch_movie_details(movie_id, api_key)
            if details:
                movies_detailed.append(details)
                if verbose == True:
                    print(f"{i + 1}: Got Details For Movie Titled: {details.get('title')}")
            time.sleep(delay)

    return movies_detailed

In [89]:
# uncomment when you want to request data
detailed_data = fetch_all_movie_details(all_movies, delay = 0.5)

1: Got Details For Movie Titled: The Shawshank Redemption
2: Got Details For Movie Titled: The Godfather
3: Got Details For Movie Titled: The Godfather Part II
4: Got Details For Movie Titled: Schindler's List
5: Got Details For Movie Titled: 12 Angry Men
6: Got Details For Movie Titled: Spirited Away
7: Got Details For Movie Titled: The Dark Knight
8: Got Details For Movie Titled: Dilwale Dulhania Le Jayenge
9: Got Details For Movie Titled: The Green Mile
10: Got Details For Movie Titled: Parasite
11: Got Details For Movie Titled: Pulp Fiction
12: Got Details For Movie Titled: Your Name.
13: Got Details For Movie Titled: The Lord of the Rings: The Return of the King
14: Got Details For Movie Titled: Forrest Gump
15: Got Details For Movie Titled: The Good, the Bad and the Ugly
16: Got Details For Movie Titled: Seven Samurai
17: Got Details For Movie Titled: GoodFellas
18: Got Details For Movie Titled: Interstellar
19: Got Details For Movie Titled: Grave of the Fireflies
20: Got Details

## Put Data into a Viewable Dataframe

In [21]:
import pandas as pd

### Movies Dataframe

In [24]:
movies_df = pd.DataFrame(all_movies)

In [26]:
movies_df.columns

Index(['adult', 'backdrop_path', 'genre_ids', 'id', 'original_language',
       'original_title', 'overview', 'popularity', 'poster_path',
       'release_date', 'title', 'video', 'vote_average', 'vote_count'],
      dtype='object')

In [28]:
movies_df = movies_df.drop(['backdrop_path', 'genre_ids', 'poster_path', 'original_title', 'video'], axis = 1)

In [30]:
movies_df.head()

,adult,id,original_language,overview,popularity,release_date,title,vote_average,vote_count
0,False,278,en,Imprisoned in the 1940s for the double murder ...,34.6551,1994-09-23,The Shawshank Redemption,8.710,28243
1,False,238,en,"Spanning the years 1945 to 1955, a chronicle o...",50.6029,1972-03-14,The Godfather,8.686,21408
2,False,240,en,In the continuing saga of the Corleone crime f...,16.7806,1974-12-20,The Godfather Part II,8.571,12930
3,False,424,en,The true story of how businessman Oskar Schind...,24.8390,1993-12-15,Schindler's List,8.565,16409
4,False,389,en,The defense and the prosecution have rested an...,23.7899,1957-04-10,12 Angry Men,8.548,9109


### Movie Details Dataframe

In [91]:
details_df = pd.DataFrame(detailed_data)

In [93]:
details_df.columns

Index(['adult', 'backdrop_path', 'belongs_to_collection', 'budget', 'genres',
       'homepage', 'id', 'imdb_id', 'origin_country', 'original_language',
       'original_title', 'overview', 'popularity', 'poster_path',
       'production_companies', 'production_countries', 'release_date',
       'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title',
       'video', 'vote_average', 'vote_count'],
      dtype='object')

In [95]:
details_df['num_spoken_languages'] = details_df['spoken_languages'].apply(lambda x: len(x) if isinstance(x, list) else None)

In [97]:
details_df['belongs_to_collection_boolean'] = details_df['belongs_to_collection'].apply(lambda x: False if x == None else True)

In [99]:
details_df['genres']= details_df['genres'].apply(lambda x: [genre['name'] for genre in x] if isinstance(x, list) else [])

In [101]:
details_df['production_companies'] = details_df['production_companies'].apply(lambda x: [company_name['name'] for company_name in x] if isinstance(x, list) else [])

In [103]:
details_df = details_df[['belongs_to_collection_boolean', 'budget', 'genres', 'id', 'origin_country', 'production_companies', 'revenue', 'runtime', 'num_spoken_languages']]

In [105]:
details_df.head()

,belongs_to_collection_boolean,budget,genres,id,origin_country,production_companies,revenue,runtime,num_spoken_languages
0,False,25000000,"[Drama, Crime]",278,[US],[Castle Rock Entertainment],28341469,142,1
1,True,6000000,"[Drama, Crime]",238,[US],"[Paramount Pictures, Alfran Productions]",245066411,175,3
2,True,13000000,"[Drama, Crime]",240,[US],"[Paramount Pictures, The Coppola Company, Amer...",102600000,202,4
3,False,22000000,"[Drama, History, War]",424,[US],[Amblin Entertainment],321365567,195,4
4,False,397751,[Drama],389,[US],"[United Artists, Orion-Nova Productions]",4360000,97,1


### Merge Dataframes and Resolve Any Duplicates

In [108]:
full_movie_list = pd.merge(movies_df, details_df, on='id', how='left')

In [110]:
full_movie_list

,adult,id,original_language,overview,popularity,release_date,title,vote_average,vote_count,belongs_to_collection_boolean,budget,genres,origin_country,production_companies,revenue,runtime,num_spoken_languages
0,False,278,en,Imprisoned in the 1940s for the double murder ...,34.6551,1994-09-23,The Shawshank Redemption,8.710,28243,False,25000000,"[Drama, Crime]",[US],[Castle Rock Entertainment],28341469,142,1
1,False,278,en,Imprisoned in the 1940s for the double murder ...,34.6551,1994-09-23,The Shawshank Redemption,8.710,28243,False,25000000,"[Drama, Crime]",[US],[Castle Rock Entertainment],28341469,142,1
2,False,238,en,"Spanning the years 1945 to 1955, a chronicle o...",50.6029,1972-03-14,The Godfather,8.686,21408,True,6000000,"[Drama, Crime]",[US],"[Paramount Pictures, Alfran Productions]",245066411,175,3
3,False,238,en,"Spanning the years 1945 to 1955, a chronicle o...",50.6029,1972-03-14,The Godfather,8.686,21408,True,6000000,"[Drama, Crime]",[US],"[Paramount Pictures, Alfran Productions]",245066411,175,3
4,False,240,en,In the continuing saga of the Corleone crime f...,16.7806,1974-12-20,The Godfather Part II,8.571,12930,True,13000000,"[Drama, Crime]",[US],"[Paramount Pictures, The Coppola Company, Amer...",102600000,202,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1087,False,267557,it,,0.2815,2014-04-30,Un fidanzato per mia moglie,5.000,110,False,0,[Comedy],[IT],"[IBC MOvie, RAI]",0,97,1
1088,False,242088,en,Years after their successful restaurant review...,0.2821,2014-04-24,The Trip to Italy,6.325,243,True,0,"[Comedy, Drama]",[GB],"[Revolution Films, Baby Cow Productions, BBC F...",0,108,2
1089,False,43648,it,A poster worker must remove the poster of the ...,0.2821,1991-12-20,The Comics 2,5.700,217,True,0,[Comedy],[IT],"[Maura International Films, Penta Film, Cecchi...",0,87,1
1090,False,13296,tr,Failed magician Iskender decides to do a tour ...,0.2822,2006-10-20,The Magician,6.900,159,False,0,"[Comedy, Drama]",[TR],[BKM Film],0,122,1


In [112]:
full_movie_list['id'].duplicated().sum()

138

In [114]:
full_movie_list = full_movie_list.drop_duplicates(subset = 'id')

In [116]:
full_movie_list['id'].duplicated().sum()

0

In [120]:
full_movie_list.to_csv("Movies.csv")